# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [17]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
#!pip install groq -q #Intalamos la librería de groq

import os
from groq import Groq #Mandar a Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('API_GROQ'))
print ("Cliente de Qroq inicializado cporrectamente.")

Cliente de Qroq inicializado cporrectamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [18]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿CUANTAS PERSONAS VIVEN EN MEXICO" #"¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

print(prompt)


¿CUANTAS PERSONAS VIVEN EN MEXICO


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [19]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt}]
    )
print(response.choices[0].message.content)


La población de México es de aproximadamente **126 millones de habitantes** (≈ 126,200,000 personas).  
Esta cifra se basa en las estimaciones interanuales del Instituto Nacional de Estadística y Geografía (INEGI) para el año 2023 y se proyecta que siga en ese rango para 2024.

Si necesitas datos más precisos (por estado, edad, género, etc.) o la fuente oficial, avísame y con gusto te los proporciono.


In [20]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-b69b02dc-7d66-41e2-ac4d-6c90ee51e8ae",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La población de México es de aproximadamente **126 millones de habitantes** (≈ 126,200,000 personas).  \nEsta cifra se basa en las estimaciones interanuales del Instituto Nacional de Estadística y Geografía (INEGI) para el año 2023 y se proyecta que siga en ese rango para 2024.\n\nSi necesitas datos más precisos (por estado, edad, género, etc.) o la fuente oficial, avísame y con gusto te los proporciono.",
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": "User: Spanish question: \"¿CUANTAS PERSONAS VIVEN EN MEXICO\". They ask: \"How many people live in Mexico\". They likely expect approximate population figure. As of 2024, Mexico's population is about 126 million. Let's provide approximate number, with lat

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [21]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

print(f"Tokens del prompt: {response.usage.prompt_tokens}")
print(f"Tokens de la respuesta: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")



Tokens del prompt: 83
Tokens de la respuesta: 216
Total tokens: 299


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [22]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time
inicio = time.time()
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt}]
    )
duracion =time.time() - inicio
#duracion_ms = duracion*1000 #para ms

print(f"Tiempo de respuesta: {duracion:.2f} segundos")
#print(f"Tiempo de respuesta: {duracion_ms:.2f} milisegundos





Tiempo de respuesta: 0.66 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [23]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role":"user", "content":prompt}]
    )
duracion_grande =time.time() - inicio

print(f"Modelo ligero: {duracion:.2f} s - {response.usage.total_tokens} tokens")
print(f"Modelo grande: {duracion_grande:.2f} s - {response_grande.usage.total_tokens} tokens")
print("\nREspuesta del modelo grande:",response_grande.choices[0].message.content)



Modelo ligero: 0.66 s - 400 tokens
Modelo grande: 0.84 s - 297 tokens

REspuesta del modelo grande: Según los datos más recientes de instituciones oficiales (INEGI) y proyecciones de la ONU, la población de México en 2024 se estima en **≈ 126 millones de habitantes** (alrededor de 126 , 2 millones).  

Esta cifra corresponde a la población total (incluyendo todas las edades) y se actualiza anualmente a partir de los censos y encuestas intercensales. Si necesitas la cifra exacta de un año específico o una proyección más detallada (por edad, sexo, distribución estatal, etc.), házmelo saber y con gusto te proporciono la información correspondiente.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [24]:
# Leer API key desde Colab Secrets

client = Groq(api_key=userdata.get('API_GROQ'))
print ("Cliente de Qroq inicializado cporrectamente.")

Cliente de Qroq inicializado cporrectamente.


In [25]:
# Definir la lista de preguntas
prompt1 = "Diferencia entre alzheimer y demencia"
prompt2 = "¿Cuántas universidades hay en México?"
prompt3 = "¿Quién es el actual presidente de Panamá"

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [26]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
#Tiempo de respuesta
inicio = time.time()
response1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt1}]
    )
respuesta1 = response1.choices[0].message.content

duracion1 =time.time() - inicio

Tokens_prompt1 = response.usage.prompt_tokens
Tokens_respuesta1 = response.usage.completion_tokens
Total_tokens1 = response.usage.total_tokens

# Guardar todo en un diccionario
resultado_1 = {
    "Respuesta": respuesta1,
    "\nTiempo_respuesta_segundos": duracion1,
    "\nTokens_prompt": Tokens_prompt1,
    "\nTokens_respuesta": Tokens_respuesta1,
    "\nTokens_totales": Total_tokens1
}

# Mostrar el resultado
print("Resultado 1:")
for clave, valor in resultado_1.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 1:
Respuesta: **Alzheimer vs. demencia**

| Aspecto | Demencia (síndrome) | Enfermedad de Alzheimer (causa específica) |
|---------|---------------------|--------------------------------------------|
| **Definición** | Trastorno cognitivo que afecta la memoria, el pensamiento, el lenguaje y la capacidad de realizar actividades cotidianas. Se describe por la presencia de deterioro cognitivo progresivo. | La forma más común de demencia. Es un trastorno neurodegenerativo en el que se acumulan placas de beta‑amilo y ovillos neurofibrilares de proteína tau en el cerebro. |
| **Causa** | Puede ser causada por distintas patologías: Alzheimer, demencia vascular, demencia frontotemporal, demencia con cuerpos de Lewy, entre otras. | Específicamente una acumulación patológica de beta‑amilo y tau; la etiología exacta incluye factores genéticos, ambientales y de estilo de vida. |
| **Etiología** | 1. **Alzheimer** 2. **Vascular** (accidentes cerebrovasculares repetidos) 3. **Frontotempora

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [27]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
#Tiempo de respuesta
inicio = time.time()
response2 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt2}]
    )
respuesta2 = response2.choices[0].message.content

duracion2 = time.time() - inicio

Tokens_prompt2 = response.usage.prompt_tokens
Tokens_respuesta2 = response.usage.completion_tokens
Total_tokens2 = response.usage.total_tokens

# Guardar todo en un diccionario
resultado_2 = {
    "Respuesta": respuesta2,
    "\nTiempo_respuesta_segundos": duracion2,
    "\nTokens_prompt": Tokens_prompt2,
    "\nTokens_respuesta": Tokens_respuesta2,
    "\nTokens_totales": Total_tokens2
}

# Mostrar el resultado
print("Resultado 2:")
for clave, valor in resultado_2.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 2:
Respuesta: En México el número de instituciones de educación superior que cumplen con los criterios de “universidad” (es decir, que ofrecen programas de licenciatura, maestría, doctorado y/o posgrado) suele rondar los **500–550**.  

| Tipo de universidad | Aproximado número (2024) |
|---------------------|--------------------------|
| **Públicas** (federales y estatales) | ≈ 270 |
| **Privadas** (con denominación de “Universidad”) | ≈ 250 |
| **Total** | ≈ 520 – 530 |

> **Fuentes principales**  
> - **ANUIES** (Asociación Nacional de Universidades e Instituciones de Educación Superior) publica cada año un listado de 539 universidades en su informe “Perfil de la Universidad Mexicana 2023”.  
> - **Secretaría de Educación Pública (SEP)**, a través del portal de Registro de Instituciones de Educación Superior, confirma la existencia de 539 universidades, de las cuales 297 son públicas y 242 privadas.  

> **Nota**  
> 1. El número exacto puede variar ligeramente según la fe

In [28]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
#Tiempo de respuesta
inicio = time.time()
response3 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role":"user", "content":prompt3}]
    )
respuesta3 = response3.choices[0].message.content

duracion3 = time.time() - inicio

Tokens_prompt3 = response.usage.prompt_tokens
Tokens_respuesta3 = response.usage.completion_tokens
Total_tokens3 = response.usage.total_tokens

# Guardar todo en un diccionario
resultado_3 = {
    "Respuesta": respuesta3,
    "\nTiempo_respuesta_segundos": duracion3,
    "\nTokens_prompt": Tokens_prompt3,
    "\nTokens_respuesta": Tokens_respuesta3,
    "\nTokens_totales": Total_tokens3
}

# Mostrar el resultado
print("Resultado 3:")
for clave, valor in resultado_3.items():
    if clave == "respuesta":
        print(f"{clave}:")
        print(valor)
    else:
        print(f"{clave}: {valor}")

Resultado 3:
Respuesta: El presidente actual de Panamá es **José Raúl Mulino**.  
- **Fecha de toma de posesión**: 1 de julio 2024.  
- **Partido político**: Partido Revolucionario Democrático (PRD).  
- **Predecesor**: Laurentino Cortizo (2019‑2024).  

Mulino asumió la presidencia tras ganar las elecciones de 2024 con una amplia mayoría. Si necesitas más detalles sobre su programa o su trayectoria política, házmelo saber.

Tiempo_respuesta_segundos: 0.3552274703979492

Tokens_prompt: 83

Tokens_respuesta: 317

Tokens_totales: 400


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [30]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)
print(resultados)

[{'Respuesta': '**Alzheimer vs. demencia**\n\n| Aspecto | Demencia (síndrome) | Enfermedad de Alzheimer (causa específica) |\n|---------|---------------------|--------------------------------------------|\n| **Definición** | Trastorno cognitivo que afecta la memoria, el pensamiento, el lenguaje y la capacidad de realizar actividades cotidianas. Se describe por la presencia de deterioro cognitivo progresivo. | La forma más común de demencia. Es un trastorno neurodegenerativo en el que se acumulan placas de beta‑amilo y ovillos neurofibrilares de proteína tau en el cerebro. |\n| **Causa** | Puede ser causada por distintas patologías: Alzheimer, demencia vascular, demencia frontotemporal, demencia con cuerpos de Lewy, entre otras. | Específicamente una acumulación patológica de beta‑amilo y tau; la etiología exacta incluye factores genéticos, ambientales y de estilo de vida. |\n| **Etiología** | 1. **Alzheimer** 2. **Vascular** (accidentes cerebrovasculares repetidos) 3. **Frontotemporal*

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [33]:
#Mostrar resultados
for resultado in resultados:
    print(resultado)

#Comprobar si resolvió correctamente las 3 preguntas
if all(resultados):
    print("El modelo ligero resolvió las 3 preguntas satisfactoriamente.")
else:
    print("El modelo ligero no resolvió las 3 preguntas satisfactoriamente.")

{'Respuesta': '**Alzheimer vs. demencia**\n\n| Aspecto | Demencia (síndrome) | Enfermedad de Alzheimer (causa específica) |\n|---------|---------------------|--------------------------------------------|\n| **Definición** | Trastorno cognitivo que afecta la memoria, el pensamiento, el lenguaje y la capacidad de realizar actividades cotidianas. Se describe por la presencia de deterioro cognitivo progresivo. | La forma más común de demencia. Es un trastorno neurodegenerativo en el que se acumulan placas de beta‑amilo y ovillos neurofibrilares de proteína tau en el cerebro. |\n| **Causa** | Puede ser causada por distintas patologías: Alzheimer, demencia vascular, demencia frontotemporal, demencia con cuerpos de Lewy, entre otras. | Específicamente una acumulación patológica de beta‑amilo y tau; la etiología exacta incluye factores genéticos, ambientales y de estilo de vida. |\n| **Etiología** | 1. **Alzheimer** 2. **Vascular** (accidentes cerebrovasculares repetidos) 3. **Frontotemporal**